# Vehicle Classification – 02 · Preprocessing and Feature Engineering

Sections 4–5 of the report:

1. Removing duplicates – and how much they would inflate cross-validation
2. Dropping the identifier
3. Missing values and encoding
4. Feature scaling
5. The test-noise model and noise-aware training (augmentation)
6. Pipeline construction
7. The validation protocol (also used in `03_Models`)
8. Feature engineering and alternative preprocessing experiments
9. Fresh-fold re-test of the most promising ones

In [1]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import norm
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (FunctionTransformer, PolynomialFeatures, PowerTransformer,
                                   QuantileTransformer, RobustScaler, SplineTransformer, StandardScaler)
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

CLASS = {0: ("Sedan", "#2a78d6"), 1: ("SUV", "#eb6834"), 2: ("Sports Car", "#1baf7a")}
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.edgecolor": AXIS, "axes.linewidth": 0.6,
    "axes.labelcolor": INK2, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "xtick.major.size": 0, "ytick.major.size": 0, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False})

The cell below loads the data and defines everything the experiments share: the de-duplicated
training set, the **measurement noise of the test file** (measured in `01_EDA`), the
augmentation helpers and the **validation protocol**. Each part is explained in the sections
that follow.

In [2]:
# ---- data -----------------------------------------------------------------------------
TARGET, ID = "vehicle_class", "vehicle_id"
train_raw = pd.read_csv("data/vehicle_train_10000.csv")
test_raw = pd.read_csv("data/vehicle_test.csv")
FEATURES = [c for c in train_raw.columns if c not in (ID, TARGET)]
train = train_raw.drop_duplicates(subset=FEATURES + [TARGET]).reset_index(drop=True)
X, y = train[FEATURES].to_numpy(float), train[TARGET].to_numpy(int)
X_test, test_ids = test_raw[FEATURES].to_numpy(float), test_raw[ID].to_numpy()
LO, HI = X.min(0), X.max(0)                                          # clip limits
DEC = np.array([0 if np.all(np.round(c) == c) else 2 for c in X.T])  # recorded precision
print(f"{len(X):,} training vehicles (after de-duplication), {len(X_test):,} test vehicles, "
      f"{len(FEATURES)} features")


# ---- the measurement noise of the test file ---------------------------------------------
def test_noise(Xv, s, share, col_sd, rng):
    """N(0, s * feature sd) on a random `share` of the values, then re-clipped to the
    limits and re-rounded to the recorded precision - the pattern of the test file."""
    if s <= 0:
        return Xv.copy()
    E = rng.normal(0.0, 1.0, Xv.shape) * (s * col_sd)
    if share < 1:
        E *= rng.random(Xv.shape) < share
    Z = np.clip(Xv + E, LO, HI)
    for j, d in enumerate(DEC):
        Z[:, j] = np.round(Z[:, j], d)
    return Z


# ---- training-time augmentation -----------------------------------------------------------
def jitter_copies(Xa, ya, sd=0.20, copies=4, seed=42):
    """Training rows + `copies` Gaussian-jittered copies (the final model's augmentation)."""
    rng = np.random.default_rng(seed)
    col_sd = Xa.std(0, ddof=1)
    parts = [Xa] + [Xa + rng.normal(0.0, sd * col_sd, Xa.shape) for _ in range(copies)]
    return np.vstack(parts), np.concatenate([ya] * (copies + 1))


def test_noise_copies(Xa, ya, s=0.15, share=1.0, copies=4, seed=0):
    """Training rows + copies carrying the test-file noise exactly (clip + round)."""
    rng = np.random.default_rng(seed)
    col_sd = Xa.std(0, ddof=1)
    parts = [Xa] + [test_noise(Xa, s, share, col_sd, rng) for _ in range(copies)]
    return np.vstack(parts), np.tile(ya, copies + 1)


def alternating_noise_copies(Xa, ya, K, seed=0):
    """K test-noise copies alternating the two calibrated readings (0.08 sd / 0.12 sd on half)."""
    rng = np.random.default_rng(seed)
    col_sd = Xa.std(0, ddof=1)
    parts = [Xa] + [test_noise(Xa, *((0.08, 1.0) if k % 2 == 0 else (0.12, 0.5)), col_sd, rng)
                    for k in range(K)]
    return np.vstack(parts), np.tile(ya, K + 1)


# ---- validation protocol --------------------------------------------------------------
VARIANTS = {            # name: (noise sd as a fraction of feature sd, share of values hit, draws)
    "clean": (0.00, 1.0, 1),
    "u08":   (0.08, 1.0, 2),     # reading 1 of the test noise: 0.08 sd on every value
    "mix12": (0.12, 0.5, 2),     # reading 2: 0.12 sd on a random half of the values
    "u15":   (0.15, 1.0, 1),     # stress level
}
SCREEN_SEEDS, CONFIRM_SEEDS = (42, 7, 123), (2024, 99, 314)


def make_folds(seeds):
    """Repeated stratified 5-fold CV; every validation fold carries its noisy copies."""
    folds = []
    for seed in seeds:
        for k, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y)):
            sd = X[tr].std(0, ddof=1)
            variants = {name: [test_noise(X[va], s, share, sd, np.random.default_rng([seed, k, vi, r]))
                               for r in range(reps)]
                        for vi, (name, (s, share, reps)) in enumerate(VARIANTS.items())}
            folds.append({"key": f"{seed}_{k}", "tr": tr, "va": va, "variants": variants})
    return folds


def proba(model, Z):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(Z)
    d = model.decision_function(Z)                      # margin models: softmax of the scores
    e = np.exp(d - d.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


P = {}      # model name -> one entry per fold: {variant: [probabilities for each noise draw]}


def run(name, fit, folds=None, store=None, n_jobs=-1):
    """fit(X_train, y_train) -> fitted model. Scores every noisy copy of every validation fold."""
    folds = FOLDS if folds is None else folds
    store = P if store is None else store

    def one(f):
        m = fit(X[f["tr"]], y[f["tr"]])
        return {v: [proba(m, Z) for Z in Zs] for v, Zs in f["variants"].items()}

    t = time.time()
    store[name] = Parallel(n_jobs=n_jobs)(delayed(one)(f) for f in folds)
    print(f"{name:<52} {time.time() - t:6.0f}s")


def fold_acc(name, folds=None, store=None):
    folds = FOLDS if folds is None else folds
    store = P if store is None else store
    out = []
    for f, probs in zip(folds, store[name]):
        yv = y[f["va"]]
        out.append({v: np.mean([(p.argmax(1) == yv).mean() for p in ps]) for v, ps in probs.items()})
    return out


BASE = "Logistic regression (final)"


def table(names, folds=None, store=None, base=BASE):
    """Accuracy (%) per noise condition and the paired gap to the final model (+- one s.e.)."""
    b = np.array([(a["u08"] + a["mix12"]) / 2 for a in fold_acc(base, folds, store)])
    rows = []
    for n in names:
        acc = fold_acc(n, folds, store)
        tl = np.array([(a["u08"] + a["mix12"]) / 2 for a in acc])
        d = (tl - b[:len(tl)]) * 100
        rows.append({"model": n, "clean": np.mean([a["clean"] for a in acc]) * 100,
                     "test-like": tl.mean() * 100, "stress 0.15": np.mean([a["u15"] for a in acc]) * 100,
                     "gap vs final": d.mean(), "+- s.e.": d.std(ddof=1) / np.sqrt(len(d))})
    return pd.DataFrame(rows).set_index("model").round(2) + 0.0      # + 0.0 turns -0.00 into 0.00


def average(names, store=None):
    """Probability average of stored models (same folds, same noisy copies)."""
    store = P if store is None else store
    first = store[names[0]]
    return [{v: [np.mean([store[n][i][v][r] for n in names], axis=0) for r in range(len(ps))]
             for v, ps in first[i].items()} for i in range(len(first))]


def stacked(f):
    return np.vstack([Z for Zs in f["variants"].values() for Z in Zs])


def unstacked(Pall, f):
    out, i, n = {}, 0, len(f["va"])
    for v, Zs in f["variants"].items():
        out[v] = []
        for _ in Zs:
            out[v].append(Pall[i:i + n])
            i += n
    return out


def logreg(C=0.1, **kw):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=20000, **kw))


class WithLDA(BaseEstimator, ClassifierMixin):
    """Appends the two LDA discriminant scores (fitted on the training rows only), so trees
    can split along the tilted class boundary instead of building a staircase."""

    def __init__(self, base):
        self.base = base

    def _tf(self, Z):
        return np.hstack([Z, self.lda_.transform(self.sc_.transform(Z))])

    def fit(self, Z, t):
        self.sc_ = StandardScaler().fit(Z)
        self.lda_ = LDA(n_components=2).fit(self.sc_.transform(Z), t)
        self.base.fit(self._tf(Z), t)
        self.classes_ = np.unique(t)
        return self

    def predict_proba(self, Z):
        return self.base.predict_proba(self._tf(Z))

    def predict(self, Z):
        return self.predict_proba(Z).argmax(1)

7,500 training vehicles (after de-duplication), 2,500 test vehicles, 20 features


## 1. Removing duplicates

2,500 of the 10,000 training rows are exact copies (measurements *and* label). If they stay,
a vehicle and its copy can land on opposite sides of a cross-validation split, so the model is
validated on rows it has memorised. We measure the effect with plain 5-fold CV.

In [3]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
X_dup, y_dup = train_raw[FEATURES].to_numpy(float), train_raw[TARGET].to_numpy()
rows = []
for name, model in {"Gaussian Naive Bayes": GaussianNB(),
                    "1-nearest-neighbour": make_pipeline(StandardScaler(), KNeighborsClassifier(1)),
                    "Decision tree (unlimited depth)": DecisionTreeClassifier(random_state=0)}.items():
    with_dup = cross_val_score(model, X_dup, y_dup, cv=cv).mean() * 100
    without = cross_val_score(model, X, y, cv=cv).mean() * 100
    rows.append({"model": name, "with duplicates": with_dup, "without": without,
                 "inflation": with_dup - without})
pd.DataFrame(rows).set_index("model").round(2)

,with duplicates,without,inflation
model,,,
Gaussian Naive Bayes,84.39,84.65,-0.26
1-nearest-neighbour,89.06,82.97,6.09
Decision tree (unlimited depth),89.61,82.29,7.32


Models that can memorise look up to **7 points better than they really are**, while Naive
Bayes (which only stores means and variances) is unaffected. Because the bias depends on the
model, it would corrupt every comparison – so the duplicates are dropped **before any split**.

## 2. Dropping the identifier

In [4]:
print("vehicle_id is unique in train:", train_raw[ID].is_unique)
print("vehicle_id values shared by train and test:", len(set(train_raw[ID]) & set(test_raw[ID])))
print("-> vehicle_id is a record number, not a vehicle property, and is never used as a feature")

vehicle_id is unique in train: True
vehicle_id values shared by train and test: 0
-> vehicle_id is a record number, not a vehicle property, and is never used as a feature


## 3. Missing values and encoding

In [5]:
print("missing values   train:", int(train_raw.isna().sum().sum()), "  test:", int(test_raw.isna().sum().sum()))
print("non-numeric features:", [f for f in FEATURES if not np.issubdtype(train_raw[f].dtype, np.number)])
print("-> no imputation or categorical encoding is needed")

missing values   train: 0   test: 0
non-numeric features: []
-> no imputation or categorical encoding is needed


## 4. Feature scaling

The features span very different units, from litres (~2) to millimetres (~4,600). A single L2
penalty on unscaled coefficients effectively switches off the large-unit features, and the
optimiser struggles.

In [6]:
display(pd.DataFrame({"min": X.min(0), "max": X.max(0), "sd": X.std(0)}, index=FEATURES).round(2).T)
rows = []
for name, model in {"raw features": LogisticRegression(C=0.1, max_iter=20000),
                    "StandardScaler": make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=20000))}.items():
    r = cross_validate(model, X, y, cv=cv, return_estimator=True)
    iters = [int(np.max(e[-1].n_iter_ if hasattr(e, "steps") else e.n_iter_)) for e in r["estimator"]]
    rows.append({"logistic regression on": name, "CV accuracy %": r["test_score"].mean() * 100,
                 "optimiser iterations": int(np.mean(iters))})
pd.DataFrame(rows).set_index("logistic regression on").round(2)

,engine_displacement_l,horsepower_hp,vehicle_weight_kg,wheelbase_mm,vehicle_length_mm,vehicle_width_mm,vehicle_height_mm,fuel_tank_capacity_l,city_fuel_efficiency_kmpl,highway_fuel_efficiency_kmpl,engine_cylinders,torque_nm,ground_clearance_mm,turning_radius_m,cargo_capacity_l,acceleration_0_100_kmh_s,braking_distance_m,tire_width_mm,drag_coefficient,battery_capacity_kwh
min,0.90,90.00,1100.00,2400.00,3900.00,1650.00,1250.00,35.00,7.00,9.00,3.0,130.00,110.00,4.10,150.00,3.50,28.00,150.00,0.22,24.00
max,4.44,550.00,2600.00,3300.00,5400.00,2050.00,2050.00,90.00,20.87,24.07,10.0,650.00,300.00,7.62,1000.00,14.00,55.00,330.00,0.42,120.00
sd,0.61,95.11,305.75,172.41,366.33,117.79,145.61,14.47,2.55,2.50,1.5,127.06,58.15,0.52,212.82,2.26,7.23,46.45,0.04,16.94


,CV accuracy %,optimiser iterations
logistic regression on,,
raw features,88.33,13235
StandardScaler,89.41,32


Scaling is applied to every scale-sensitive model (logistic regression, LDA, SVM, neural
networks). Tree models use the raw values, since splits are unaffected by scaling.

## 5. The test-noise model and noise-aware training

`test_noise()` (defined in the setup cell) reproduces the measurement error found in `01_EDA`:
Gaussian error on part of the values, then re-clipping and re-rounding exactly like the test
file. The final model is trained on the clean rows **plus four jittered copies** (0.20 sd per
feature), which teaches it that a vehicle keeps its class when its measurements wobble.

In [7]:
rng = np.random.default_rng(0)
example = test_noise(X[:5], 0.12, 0.5, X.std(0, ddof=1), rng)
cols = ["engine_displacement_l", "horsepower_hp", "vehicle_weight_kg", "ground_clearance_mm", "braking_distance_m"]
idx = [FEATURES.index(c) for c in cols]
display(pd.concat({"original": pd.DataFrame(X[:5][:, idx], columns=cols),
                   "with test-like noise": pd.DataFrame(example[:, idx], columns=cols)}, axis=1))
Xa, ya = jitter_copies(X, y)
print(f"augmented training set: {Xa.shape[0]:,} rows ({len(X):,} real + {Xa.shape[0] - len(X):,} jittered)")

original                                                                         with test-like noise                \
  engine_displacement_l horsepower_hp vehicle_weight_kg ground_clearance_mm braking_distance_m engine_displacement_l horsepower_hp   
0                  2.56         290.0            1532.0               197.0               43.0                  2.56         290.0   
1                  2.70         350.0            1529.0               251.0               41.0                  2.70         350.0   
2                  2.50         335.0            1100.0               110.0               29.0                  2.50         352.0   
3                  3.00         345.0            2069.0               168.0               51.0                  2.97         345.0   
4                  1.59         155.0            1574.0               179.0               47.0                  1.60         155.0   

                                                            
  vehicle_weight_kg ground_clearance_mm braking_distance_m  
0            1555.0               197.0               43.0  
1            1505.0               250.0               41.0  
2            1149.0               110.0               29.0  
3            2069.0               168.0               51.0  
4            1574.0               179.0               47.0

augmented training set: 37,500 rows (7,500 real + 30,000 jittered)


## 6. Pipeline construction

Every model is a `Pipeline`, so the scaler – and any projection or augmentation – is fitted on
the training fold only, identically for training, validation and test data, with no leakage.

In [8]:
final_pipeline = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=20000))
final_pipeline.fit(*jitter_copies(X, y))
print(final_pipeline)
print(f"accuracy on the clean training rows: {(final_pipeline.predict(X) == y).mean():.4f}")

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(C=0.1, max_iter=20000))])
accuracy on the clean training rows: 0.8967


## 7. The validation protocol

A single 80/20 split is too noisy here: at about 89% accuracy on 1,500 rows the standard error
is ~0.8 points, larger than most differences between models. So:

* **Repeated stratified 5-fold CV** – 3 repeats = 15 folds of 6,000 / 1,500 rows.
* **Test-like scoring** – each validation fold is corrupted with the measured noise in its two
  readings (0.08 sd on every value; 0.12 sd on a random half), re-clipped and re-rounded.
  *Test-like accuracy* is the mean of the two; a 0.15 sd *stress* level is also reported.
* **Paired comparisons** – every model sees the same folds and the same noisy copies, so each
  gap to the final model has its own standard error.
* **Confirmation on 15 fresh folds** – the best of many screened ideas is inflated by luck
  (the *winner's curse*), so promising ideas are re-tested on new folds.

In [9]:
FOLDS, CONFIRM = make_folds(SCREEN_SEEDS), make_folds(CONFIRM_SEEDS)
print(f"{len(FOLDS)} screening folds and {len(CONFIRM)} confirmation folds; each validation fold has "
      f"{sum(len(z) for z in FOLDS[0]['variants'].values())} noisy copies of {len(FOLDS[0]['va'])} rows")
run(BASE, lambda Xt, yt: logreg(0.1).fit(*jitter_copies(Xt, yt)))
table([BASE])

15 screening folds and 15 confirmation folds; each validation fold has 6 noisy copies of 1500 rows


Logistic regression (final)                               4s


,clean,test-like,stress 0.15,gap vs final,+- s.e.
model,,,,,
Logistic regression (final),89.58,89.26,88.96,0.0,0.0


## 8. Feature engineering and alternative preprocessing

Every idea is placed in front of the final logistic regression and scored with the protocol
above:

* **New features** – physical ratios, polynomial interactions, spline expansions, clip-limit flags.
* **Transformations** – Yeo-Johnson and rank-Gauss (to make skewed features more Gaussian),
  robust scaling.
* **Un-censoring** – each value sitting exactly on a clip limit is replaced by the expected
  true value beyond the limit, predicted from the other 19 features (a truncated-normal mean).
* **Denoising and selection** – dropping small principal components, ANOVA feature selection,
  lasso, and removing training rows the model itself finds implausible.
* **Noise-aware training** with 10 copies that reproduce the test noise exactly.

In [10]:
F_IDX = {n: i for i, n in enumerate(FEATURES)}

def add_ratios(Z):
    g = lambda n: Z[:, F_IDX[n]]
    volume = g("vehicle_length_mm") * g("vehicle_width_mm") * g("vehicle_height_mm") / 1e9
    return np.hstack([Z, np.column_stack([
        g("horsepower_hp") / g("vehicle_weight_kg"), g("torque_nm") / g("vehicle_weight_kg"),
        g("horsepower_hp") / g("engine_displacement_l"), g("engine_displacement_l") / g("engine_cylinders"),
        volume, g("cargo_capacity_l") / volume, g("battery_capacity_kwh") / g("vehicle_weight_kg"),
        g("vehicle_length_mm") / g("vehicle_width_mm"), g("vehicle_height_mm") - g("ground_clearance_mm"),
        g("highway_fuel_efficiency_kmpl") / g("city_fuel_efficiency_kmpl")])])

def add_clip_flags(Z):
    flags = [np.isclose(Z[:, j], b).astype(float) for j in range(X.shape[1]) for b in (LO[j], HI[j])
             if np.mean(X[:, j] == b) > 0.01]
    return np.hstack([Z, np.column_stack(flags)])

class Uncensor(BaseEstimator, TransformerMixin):
    """Values exactly on a clip limit -> expected hidden value beyond the limit (truncated-normal
    mean around a linear prediction from the other features, fitted on unclipped rows)."""
    def fit(self, Xa, ya=None):
        self.jobs_ = []
        for j in range(Xa.shape[1]):
            at = (Xa[:, j] == LO[j]) | (Xa[:, j] == HI[j])
            if max(np.mean(Xa[:, j] == LO[j]), np.mean(Xa[:, j] == HI[j])) < 0.01:
                continue
            others = np.delete(np.arange(Xa.shape[1]), j)
            reg = LinearRegression().fit(Xa[~at][:, others], Xa[~at, j])
            self.jobs_.append((j, others, reg, np.std(Xa[~at, j] - reg.predict(Xa[~at][:, others]))))
        return self
    def transform(self, Z):
        Z = Z.copy()
        for j, others, reg, s in self.jobs_:
            m = reg.predict(Z[:, others])
            lo, hi = Z[:, j] == LO[j], Z[:, j] == HI[j]
            a, b = (LO[j] - m[lo]) / s, (HI[j] - m[hi]) / s
            Z[lo, j] = m[lo] - s * norm.pdf(a) / np.maximum(norm.cdf(a), 1e-12)
            Z[hi, j] = m[hi] + s * norm.pdf(b) / np.maximum(norm.sf(b), 1e-12)
        return Z

class CleanThenFit(BaseEstimator, ClassifierMixin):
    """Drop training rows whose out-of-fold probability for their own label is below tau,
    then fit the usual noise-augmented logistic regression on the rest."""
    def __init__(self, tau=0.05):
        self.tau = tau
    def fit(self, Xa, ya):
        Pf = cross_val_predict(logreg(0.1), Xa, ya, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                               method="predict_proba")
        keep = Pf[np.arange(len(ya)), ya] >= self.tau
        self.m_ = logreg(0.1).fit(*jitter_copies(Xa[keep], ya[keep]))
        self.classes_ = self.m_.classes_
        return self
    def predict_proba(self, Z):
        return self.m_.predict_proba(Z)

lr_on = lambda *steps, C=0.1, **kw: make_pipeline(*steps, LogisticRegression(C=C, max_iter=20000, **kw))
EXPERIMENTS = {
    "10 physics ratios (power/weight, hp per litre, ...)":
        lambda Xt, yt: lr_on(FunctionTransformer(add_ratios), StandardScaler()).fit(*jitter_copies(Xt, yt)),
    "Degree-2 polynomial features (230)":
        lambda Xt, yt: lr_on(PolynomialFeatures(2, include_bias=False), StandardScaler(), C=0.01).fit(*jitter_copies(Xt, yt)),
    "Spline expansion (GAM, 5 knots)":
        lambda Xt, yt: lr_on(SplineTransformer(n_knots=5, degree=3, extrapolation="linear"), StandardScaler(),
                             C=0.05).fit(*jitter_copies(Xt, yt)),
    "Clip-limit indicator flags":
        lambda Xt, yt: lr_on(FunctionTransformer(add_clip_flags), StandardScaler()).fit(
            *test_noise_copies(Xt, yt, 0.12, 0.5)),
    "Yeo-Johnson power transform":
        lambda Xt, yt: lr_on(PowerTransformer(), C=0.3).fit(*jitter_copies(Xt, yt)),
    "Rank-Gauss (quantile) transform":
        lambda Xt, yt: lr_on(QuantileTransformer(n_quantiles=1000, output_distribution="normal", random_state=0),
                             StandardScaler()).fit(*jitter_copies(Xt, yt)),
    "Robust scaling (median / IQR)":
        lambda Xt, yt: lr_on(RobustScaler()).fit(*jitter_copies(Xt, yt)),
    "Un-censoring clipped values":
        lambda Xt, yt: lr_on(Uncensor(), StandardScaler()).fit(*jitter_copies(Xt, yt)),
    "PCA denoising (17 components)":
        lambda Xt, yt: lr_on(StandardScaler(), PCA(17)).fit(*jitter_copies(Xt, yt)),
    "Feature selection (top 17 by ANOVA)":
        lambda Xt, yt: lr_on(StandardScaler(), SelectKBest(f_classif, k=17)).fit(*jitter_copies(Xt, yt)),
    "Lasso (L1) logistic regression":
        lambda Xt, yt: lr_on(StandardScaler(), C=0.05, penalty="l1", solver="saga", tol=1e-3).fit(*jitter_copies(Xt, yt)),
    "Data cleaning (drop implausible rows)":
        lambda Xt, yt: CleanThenFit(0.05).fit(Xt, yt),
    "Training on 10 test-noise copies":
        lambda Xt, yt: logreg(0.1).fit(*alternating_noise_copies(Xt, yt, 10)),
}
for name, fit in EXPERIMENTS.items():
    run(name, fit)

10 physics ratios (power/weight, hp per litre, ...)       4s


Degree-2 polynomial features (230)                        9s


Spline expansion (GAM, 5 knots)                           6s


Clip-limit indicator flags                                2s


Yeo-Johnson power transform                               9s


...

PCA denoising (17 components)                             2s


Feature selection (top 17 by ANOVA)                       2s


Lasso (L1) logistic regression                            2s


Data cleaning (drop implausible rows)                     2s


Training on 10 test-noise copies                          2s


In [11]:
fe = table([BASE] + list(EXPERIMENTS))
fe

,clean,test-like,stress 0.15,gap vs final,+- s.e.
model,,,,,
Logistic regression (final),89.58,89.26,88.96,0.00,0.00
"10 physics ratios (power/weight, hp per litre, ...)",89.45,89.24,88.96,-0.02,0.02
Degree-2 polynomial features (230),89.42,89.20,88.95,-0.06,0.03
"Spline expansion (GAM, 5 knots)",89.37,89.12,88.76,-0.15,0.06
Clip-limit indicator flags,89.46,89.21,88.86,-0.06,0.04
Yeo-Johnson power transform,89.61,89.32,88.95,0.06,0.02
Rank-Gauss (quantile) transform,89.48,89.27,89.01,0.01,0.04
Robust scaling (median / IQR),89.57,89.27,88.94,0.01,0.01
Un-censoring clipped values,89.59,89.28,88.97,0.02,0.01


## 9. Fresh-fold re-test

The ideas with a positive or near-zero screening gap are re-run on the **15 confirmation
folds**, which nothing was tuned on.

In [12]:
PC = {}
for name in [BASE, "Degree-2 polynomial features (230)", "Yeo-Johnson power transform",
             "Un-censoring clipped values"]:
    fit = (lambda Xt, yt: logreg(0.1).fit(*jitter_copies(Xt, yt))) if name == BASE else EXPERIMENTS[name]
    run(name, fit, folds=CONFIRM, store=PC)
table(list(PC), folds=CONFIRM, store=PC)

Logistic regression (final)                               1s


Degree-2 polynomial features (230)                        9s


Yeo-Johnson power transform                               9s


Un-censoring clipped values                               2s


,clean,test-like,stress 0.15,gap vs final,+- s.e.
model,,,,,
Logistic regression (final),89.34,89.20,88.78,0.00,0.00
Degree-2 polynomial features (230),89.35,89.19,88.75,-0.01,0.03
Yeo-Johnson power transform,89.32,89.20,88.81,0.00,0.02
Un-censoring clipped values,89.40,89.21,88.80,0.01,0.02


**Result:** none of the ideas improves the model. The Yeo-Johnson transform is a useful lesson:
it looked like a win in screening (about +0.06 points, roughly three standard errors) but is an
exact tie on the fresh folds – the winner's curse that the confirmation step exists to catch.
The standardised raw features are already the right representation for a linear boundary.

*(Lasso uses the stochastic `saga` solver, so its last decimal can vary slightly between runs.)*